# Experimento 5 — Remoção das Variáveis Categóricas

O quinto experimento remove as variáveis categóricas `Country` e
`Waterbody Type` do conjunto de entrada, mantendo apenas variáveis
físico-químicas numéricas:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Nitrogen (mg/l)`

Nos experimentos anteriores, as variáveis categóricas forneciam ao
modelo informações contextuais externas — padrões específicos de países
e tipos de corpos hídricos que podem influenciar a qualidade da água
independentemente de seus parâmetros físico-químicos. A remoção dessas
variáveis força o LightGBM a encontrar relações diretamente entre os
parâmetros mensuráveis da água e as classificações do `conama_status`,
sem depender de contexto geográfico ou estrutural.

Esse cenário é relevante do ponto de vista prático: em situações reais
de monitoramento hídrico, nem sempre é possível ou relevante categorizar
a amostra por país ou tipo de corpo hídrico. Um modelo capaz de
classificar a qualidade da água apenas com base em parâmetros
físico-químicos possui maior potencial de generalização para novos
pontos de monitoramento.

O objetivo deste experimento é investigar:

- quanto as variáveis categóricas contribuíam para o desempenho
  observado nos experimentos anteriores;
- se o LightGBM consegue manter capacidade preditiva relevante
  utilizando apenas variáveis numéricas;
- como o comportamento do LightGBM se compara ao Random Forest
  nas mesmas condições de entrada.

Nenhuma técnica de balanceamento foi aplicada, permitindo observar
exclusivamente o impacto da remoção das variáveis categóricas no
comportamento do LightGBM.

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada utilizando `train_test_split` com
`random_state=42` e `stratify=y`, mantendo os mesmos parâmetros
de todos os experimentos anteriores.

Após o treinamento, o modelo foi avaliado utilizando:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusão

## Preparação do ambiente

In [1]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

## Definição de X e y

Em relação aos experimentos anteriores, as variáveis categóricas
`Country` e `Waterbody Type` foram removidas. O `ColumnTransformer`
com `OneHotEncoder` também não é necessário, pois todas as variáveis
de entrada são numéricas — o Pipeline passa diretamente para o
classificador.

In [4]:
# Definição de X e y
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [5]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 3)
Teste: (28280, 3)


In [10]:
model = Pipeline(
    steps=[
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi
realizada a avaliação das métricas no conjunto de treino, permitindo
comparar o comportamento entre treino e teste e identificar possíveis
variações no overfitting em relação aos experimentos anteriores.

In [12]:
# Métricas de treino
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:", train_accuracy)
print("Train Precision:", train_precision)
print("Train Recall:", train_recall)
print("Train F1:", train_f1)
print("Train Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy: 0.7852173374941434
Train Precision: 0.755081382109715
Train Recall: 0.7852173374941434
Train F1: 0.7527906693437032
Train Confusion Matrix:
 [[ 1522  1892    20  4126]
 [  604  7682    30 13362]
 [  125   582   111   288]
 [  486  2757    24 79508]]


In [13]:
# Métricas de teste
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7781471004243281

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.48      0.18      0.26      1890
         Boa       0.57      0.35      0.43      5419
     Crítica       0.22      0.03      0.05       277
   Excelente       0.82      0.96      0.88     20694

    accuracy                           0.78     28280
   macro avg       0.52      0.38      0.40     28280
weighted avg       0.74      0.78      0.74     28280


Confusion Matrix:
[[  332   508    11  1039]
 [  182  1876     6  3355]
 [   34   161     7    75]
 [  148   747     8 19791]]


## Resultados Obtidos — Experimento 5

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.778
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

```python
0.785
```

### Comparação com os Experimentos 1 e 4
*LightGBM 4 variáveis vs LightGBM 5 variáveis vs LightGBM 3 variáveis— sem balanceamento*

| Métrica             | Exp 1 — LGBM 4 var | Exp 4 — LGBM 5 var | Exp 5 — LGBM 3 var |
|---------------------|--------------------|--------------------|--------------------|
| Accuracy treino     | 0.750              | 0.683              | 0.785              |
| Accuracy teste      | 0.745              | 0.673              | 0.778              |
| Overfitting         | 0.005              | 0.010              | 0.007              |
| Precision Atenção   | 0.42               | 0.25               | 0.48               |
| Recall Atenção      | 0.21               | 0.61               | 0.18               |
| F1 Atenção          | 0.28               | 0.35               | 0.26               |
| Precision Boa       | 0.39               | 0.45               | 0.57               |
| Recall Boa          | 0.16               | 0.40               | 0.35               |
| F1 Boa              | 0.23               | 0.42               | 0.43               |
| Precision Crítica   | 0.17               | 0.09               | 0.22               |
| Recall Crítica      | 0.01               | 0.67               | 0.03               |
| F1 Crítica          | 0.02               | 0.16               | 0.05               |
| Precision Excelente | 0.79               | 0.92               | 0.82               |
| Recall Excelente    | 0.96               | 0.75               | 0.96               |
| F1 Excelente        | 0.87               | 0.83               | 0.88               |


### Comparação entre algoritmos — 3 variáveis numéricas sem balanceamento
*Random Forest (Experimento 5) vs LightGBM (Experimento 5)*

| Métrica             | RF 3 var s/ bal | LGBM 3 var s/ bal |
|---------------------|-----------------|-------------------|
| Accuracy treino     | 0.923           | 0.785             |
| Accuracy teste      | 0.763           | 0.778             |
| Overfitting         | 0.160           | 0.007             |
| Precision Atenção   | 0.39            | 0.48              |
| Precision Boa       | 0.54            | 0.57              |
| Precision Crítica   | 0.13            | 0.22              |
| Precision Excelente | 0.81            | 0.82              |
| Recall Atenção      | 0.18            | 0.18              |
| Recall Boa          | 0.32            | 0.35              |
| Recall Crítica      | 0.04            | 0.03              |
| Recall Excelente    | 0.94            | 0.96              |
| F1 Atenção          | 0.24            | 0.26              |
| F1 Boa              | 0.40            | 0.43              |
| F1 Crítica          | 0.06            | 0.05              |
| F1 Excelente        | 0.87            | 0.88              |

### Conclusão

## Análise Comparativa dos Experimentos (1, 4 e 5)

No Experimento 1, com 4 variáveis, o modelo apresenta um comportamento relativamente estável, com acurácia de treino e teste muito próximas (0.750 e 0.745) e um overfitting praticamente inexistente (0.005). Esse resultado sugere um modelo bem ajustado do ponto de vista estatístico. No entanto, essa estabilidade esconde um problema importante: a incapacidade de identificar a classe `Crítica`, cujo recall é praticamente zero (0.01). Na prática, o modelo funciona bem para prever a classe dominante, mas falha em capturar eventos raros e relevantes.

No Experimento 4, com 5 variáveis, ocorre uma mudança significativa nesse comportamento. A acurácia geral cai, indicando um modelo menos eficiente do ponto de vista global, mas há um ganho expressivo na sensibilidade às classes minoritárias. O destaque é a classe `Crítica`, cujo recall sobe para 0.67, representando uma mudança estrutural importante: o modelo passa a enxergar padrões que antes ignorava. Esse ganho, no entanto, vem acompanhado de maior instabilidade e perda de desempenho em algumas classes, especialmente na precisão.

Já no Experimento 5, com apenas 3 variáveis, o modelo atinge o melhor desempenho em termos de acurácia e também o menor overfitting entre todos os cenários. Isso indica uma generalização mais eficiente e um modelo mais “limpo” em termos estatísticos. Porém, esse ganho de estabilidade tem um custo alto: a perda quase total da capacidade de identificar a classe `Crítica`, cujo recall cai para 0.03. Ou seja, o modelo se torna mais confiável no geral, mas menos útil para detectar eventos raros.

A classe `Atenção` também reflete bem esse trade-off. No Experimento 4, ela ganha sensibilidade (recall alto), mas perde precisão. No Experimento 5, ocorre o inverso: a precisão melhora, mas o recall despenca. Isso mostra que o modelo alterna entre “alertar mais” ou “errar menos”, dependendo da complexidade das variáveis.

A classe `Boa` apresenta o comportamento mais consistente entre os experimentos, com evolução gradual e leve melhora no Experimento 5. Já a classe `Excelente` permanece dominante em todos os cenários, com desempenho sempre alto, o que reforça o viés natural do modelo em direção à classe majoritária.

Em síntese, o Experimento 1 privilegia estabilidade, o Experimento 4 privilegia sensibilidade às classes minoritárias, e o Experimento 5 privilegia acurácia e generalização.

---

## Comparação com Random Forest

O Random Forest apresentou forte overfitting (0.160), com grande diferença entre treino e teste, indicando tendência a memorização.

O LightGBM, por outro lado, apresentou overfitting muito baixo (0.007), com desempenho mais consistente entre treino e teste.

Apesar disso, ambos os modelos apresentam dificuldade na classe `Crítica`, mostrando que o problema principal não está apenas no algoritmo, mas na ausência de estratégias de balanceamento.


In [14]:
# Salvamento de Resultados
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

resultados = {
    "experimento": "exp_05_lightgbm_3var_sem_balanceamento",
    "algoritmo": "LightGBM",
    "balanceamento": "nenhum",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas.")

Métricas salvas.
